# EMG Signal Preprocessing Pipeline

Based on `1.0_analyze.ipynb` and `1.0_signal-verification.ipynb`

**Preprocessing steps:**
1. 5s Baseline correction
2. Bandpass filter (20-500 Hz)
3. Notch filter at 50 Hz
4. Segment signal according to event markers
5. Compute Power Spectral Density (PSD)
6. Extract frequency-domain features (1-450 Hz for each channel)

**Output:** `dataset_filtered_1_450.csv` compatible with `dt_classification_grid_search.ipynb`

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, iirnotch, welch

In [ ]:
# Configuration
DATA_DIR = "data"
OUTPUT_PATH = "dataset_filtered_1_450.csv"
SAMPLE_RATE = 1000  # Hz
BASELINE_DURATION = 5.0  # seconds
BANDPASS_LOW = 20  # Hz
BANDPASS_HIGH = 500  # Hz
NOTCH_FREQ = 50  # Hz
FREQ_RANGE = (1, 450)  # Hz for PSD features
N_CHANNELS = 3
REST_TIME_MAP = {"1min": 0, "5min": 1, "10min": 2}

In [ ]:
def load_and_preprocess(
    filepath: str,
    sfreq: int = SAMPLE_RATE,
    baseline_duration: float = BASELINE_DURATION,
    bandpass_low: int = BANDPASS_LOW,
    bandpass_high: int = BANDPASS_HIGH,
    notch_freq: int = NOTCH_FREQ,
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """
    Load EMG data, apply baseline correction, bandpass and notch filters.
    Returns: (time, filtered_data, points)
    """
            df = pd.read_csv(f"{filepath}.csv")
    time = df.iloc[:, 0].values  # First column: time
    data = df.iloc[:, 1:4].values.astype(float)  # emg1, emg2, emg3
    
    # Convert from raw ADC if needed (values 0-65535)
    if np.max(data) > 10:
        data = data * 3.3 / 65535
    
    with open(f"{filepath}_points.json", "r") as f:
        points = json.load(f)
    
    # 1. Baseline correction (5s): use data before baseline marker
    baseline_end = points["baseline"]
    baseline_idx = np.searchsorted(time, baseline_end)
    baseline_idx = min(baseline_idx, len(time) - 1)
    baseline_mean = np.mean(data[:baseline_idx], axis=0)
    data = data - baseline_mean
    
    # Crop to signal after baseline (up to ~22s as in original)
    end_idx = np.searchsorted(time, 22.0)
    if end_idx >= len(time):
        end_idx = len(time)
    time = time[baseline_idx:end_idx]
    data = data[baseline_idx:end_idx, :]
    
    # Resample to uniform sampling if needed (for Welch method)
    duration = time[-1] - time[0]
    n_samples = int(duration * sfreq)
    if abs(len(time) - n_samples) > n_samples * 0.1:  # >10% difference
        from scipy.interpolate import interp1d
        time_uniform = np.linspace(time[0], time[-1], n_samples)
        data_resampled = np.zeros((n_samples, N_CHANNELS))
        for ch in range(N_CHANNELS):
            f_interp = interp1d(time, data[:, ch], kind='linear', fill_value='extrapolate')
            data_resampled[:, ch] = f_interp(time_uniform)
        time, data = time_uniform, data_resampled
    
    # 2. Bandpass filter (20-500 Hz)
    nyq = sfreq / 2
    low = bandpass_low / nyq
    high = min(bandpass_high / nyq, 0.99)
    b, a = butter(4, [low, high], btype="bandpass")
    for ch in range(N_CHANNELS):
        data[:, ch] = filtfilt(b, a, data[:, ch])
    
    # 3. Notch filter at 50 Hz
    b_notch, a_notch = iirnotch(notch_freq, 30.0, sfreq)
    for ch in range(N_CHANNELS):
        data[:, ch] = filtfilt(b_notch, a_notch, data[:, ch])
    
    return time, data, points

In [ ]:
def segment_signal(
    time: np.ndarray,
    data: np.ndarray,
    points: Dict,
) -> List[Tuple[np.ndarray, np.ndarray, str]]:
    """
    Segment signal according to event markers.
    Returns list of (time_seg, data_seg, label) for each segment.
    Segments: close_0->open_0 (sit), open_0->close_1 (stand), close_1->open_1, ...
    """
    segments = []
    for i in range(3):
        t_start = points[f"close_{i}"]
        t_end = points[f"open_{i}"]
        idx_start = np.searchsorted(time, t_start)
        idx_end = np.searchsorted(time, t_end)
        if idx_start < idx_end:
            seg_time = time[idx_start:idx_end]
            seg_data = data[idx_start:idx_end, :]
            segments.append((seg_time, seg_data, f"sit_{i}"))
        
        if i < 2:
            t_start = points[f"open_{i}"]
            t_end = points[f"close_{i+1}"]
            idx_start = np.searchsorted(time, t_start)
            idx_end = np.searchsorted(time, t_end)
            if idx_start < idx_end:
                seg_time = time[idx_start:idx_end]
                seg_data = data[idx_start:idx_end, :]
                segments.append((seg_time, seg_data, f"stand_{i}"))
    return segments

In [ ]:
def compute_psd_features(
    data_seg: np.ndarray,
    sfreq: int = SAMPLE_RATE,
    freq_range: Tuple[int, int] = FREQ_RANGE,
) -> Dict[str, float]:
    """
    Compute PSD using Welch method and extract features for 1-450 Hz per channel.
    Returns dict with keys ch_{channel}_{freq_hz}.
    """
    nperseg = min(256, len(data_seg) // 2)  # Ensure valid segment length
    if nperseg < 8:
        return None
    
    features = {}
    for ch in range(N_CHANNELS):
        f, psd = welch(data_seg[:, ch], fs=sfreq, nperseg=nperseg)
        for i, freq in enumerate(f):
            freq_int = int(round(freq))
            if freq_range[0] <= freq_int <= freq_range[1]:
                features[f"ch_{ch+1}_{freq_int}"] = psd[i]
    
    # Ensure we have all 450 frequencies per channel (interpolate if needed)
    result = {}
    for ch in range(1, N_CHANNELS + 1):
        for freq in range(freq_range[0], freq_range[1] + 1):
            key = f"ch_{ch}_{freq}"
            if key in features:
                result[key] = features[key]
            else:
                # Find nearest frequency
                nearest = min(features.keys(), 
                    key=lambda k: abs(int(k.split('_')[-1]) - freq) if k.startswith(f'ch_{ch}_') else 999)
                if nearest.startswith(f'ch_{ch}_'):
                    result[key] = features[nearest]
                else:
                    result[key] = 0.0
    return result

In [ ]:
def compute_psd_features_v2(
    data_seg: np.ndarray,
    sfreq: int = SAMPLE_RATE,
    freq_range: Tuple[int, int] = FREQ_RANGE,
) -> Dict[str, float]:
    """
    Compute PSD using Welch and extract 1-450 Hz features per channel.
    Maps Welch output frequencies to integer Hz bins.
    """
    nperseg = min(512, len(data_seg) // 2)
    if nperseg < 8:
        return None
    
    result = {}
    for ch in range(N_CHANNELS):
        f, psd = welch(data_seg[:, ch], fs=sfreq, nperseg=nperseg)
        for freq_hz in range(freq_range[0], freq_range[1] + 1):
            idx = np.argmin(np.abs(f - freq_hz))
            result[f"ch_{ch+1}_{freq_hz}"] = float(psd[idx])
    return result

In [ ]:
def process_all_data(data_dir: str = DATA_DIR) -> pd.DataFrame:
    """Process all EMG files and build dataset."""
    rows = []
    skipped = []
    
    for user_id in sorted(os.listdir(data_dir)):
        user_path = Path(data_dir) / user_id
        if not user_path.is_dir() or not user_id.isdigit():
            continue
        
        for rest_time in ["1min", "5min", "10min"]:
            rest_path = user_path / rest_time
            if not rest_path.exists():
                continue
            
            rest_int = REST_TIME_MAP[rest_time]
            
            for exp_num in [1, 2, 3]:
                csv_file = rest_path / f"exp_{exp_num}.csv"
                json_file = rest_path / f"exp_{exp_num}_points.json"
                if "broken" in str(csv_file):
                    continue
                if not csv_file.exists() or not json_file.exists():
                    skipped.append(str(csv_file))
                    continue
                filepath = str(rest_path / f"exp_{exp_num}")
                
                try:
                    time, data, points = load_and_preprocess(str(filepath))
                    segments = segment_signal(time, data, points)
                    
                    for seq, (seg_time, seg_data, label) in enumerate(segments):
                        features = compute_psd_features_v2(seg_data)
                        if features is None:
                            continue
                        row = {
                            "user_id": int(user_id),
                            "rest_int": rest_int,
                            "exp": exp_num,
                            "seq": seq,
                            **features
                        }
                        rows.append(row)
                except Exception as e:
                    skipped.append(f"{filepath}: {e}")
    
    if skipped:
        print(f"Skipped {len(skipped)} files:", skipped[:5], "...")
    
    return pd.DataFrame(rows)

In [ ]:
# Run preprocessing pipeline
print("Processing EMG data...")
df = process_all_data()
print(f"Processed {len(df)} samples")
print(f"Columns: {list(df.columns[:10])}... ({len(df.columns)} total)")
df.head()

In [ ]:
# Save preprocessed dataset
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Shape: {df.shape}")
print(f"Feature columns (excluding user_id, rest_int, exp, seq): {len(df.columns) - 4}")

In [ ]:
# Verify compatibility with dt_classification_grid_search
REMOVE_LIST = ["user_id", "rest_int", "exp", "seq"]
features_list = [c for c in df.columns if c not in REMOVE_LIST]
print(f"Number of features: {len(features_list)}")
print(f"Expected: 1350 (3 channels x 450 frequencies)")
print(f"rest_int unique values: {sorted(df['rest_int'].unique())}")

In [ ]:
# Quick verification: load output and check compatibility with dt_classification_grid_search
try:
    check_df = pd.read_csv(OUTPUT_PATH)
    remove_list = ["user_id", "rest_int", "exp", "seq"]
    features_list = [c for c in check_df.columns if c not in remove_list]
    print("Verification passed!")
    print(f"  - Rows: {len(check_df)}")
    print(f"  - Features: {len(features_list)} (expected 1350)")
    print(f"  - rest_int values: {sorted(check_df['rest_int'].unique())}")
    print(f"  - Sample feature names: {features_list[:5]} ... {features_list[-3:]}")
except FileNotFoundError:
    print("Run the pipeline above first to generate", OUTPUT_PATH)